# ComplianceGPT — Single Run (Dev)

This notebook supports **freeform** and **gold-set** single runs.

- Optional precomputed QUR rewrites
- Optional verifier (only when `gold_row` is present)

Use this notebook for development/debugging, not for committee demos.

In [ ]:
# --- Colab + Drive bootstrap ---
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"
assert os.path.isdir(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import compliancegpt
print("compliancegpt.__file__ =", compliancegpt.__file__)


In [ ]:
# --- Canonical paths (Drive) ---
import os

DATA_DIR = os.path.join(PROJECT_ROOT, "data")

REV5_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev5_catalog.jsonl")
REV4_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev4_catalog.jsonl")

REV5_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev5_gold-set_100q.csv")
REV4_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev4_gold-set_36q.csv")

QUR_REV5_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev5.csv")
QUR_REV4_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev4.csv")

ORG_PROFILE_REV5 = os.path.join(DATA_DIR, "ODP", "rev5", "org_profile_example_rev5.yaml")
ORG_PROFILE_REV4 = os.path.join(DATA_DIR, "ODP", "rev4", "org_profile_example_rev4.yaml")


In [ ]:
# --- Helpers: load gold sets and QUR rewrite maps ---
from typing import Any, Dict, List, Optional
import pandas as pd

def load_gold(framework_version: str) -> pd.DataFrame:
    path = REV5_GOLD_PATH if framework_version.lower() == "rev5" else REV4_GOLD_PATH
    df = pd.read_csv(path, encoding="latin1")
    if "id" in df.columns and "query_id" not in df.columns:
        df["query_id"] = df["id"].astype(str)
    if "question" in df.columns and "original_query" not in df.columns:
        df["original_query"] = df["question"].astype(str)
    return df

def load_qur_map(framework_version: str) -> Dict[str, List[str]]:
    path = QUR_REV5_PATH if framework_version.lower() == "rev5" else QUR_REV4_PATH
    if not os.path.exists(path):
        return {}
    df = pd.read_csv(path)
    required = {"query_id", "rewrite_rank", "rewritten_query"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"Unexpected QUR CSV schema. Missing {required - set(df.columns)}. Columns={list(df.columns)}")
    out: Dict[str, List[str]] = {}
    for qid, g in df.groupby(df["query_id"].astype(str)):
        g2 = g.sort_values("rewrite_rank")
        out[qid] = [str(x) for x in g2["rewritten_query"].tolist() if isinstance(x, str) and x.strip()]
    return out

def get_gold_row(df: pd.DataFrame, qid: Any) -> Dict[str, Any]:
    qid_str = str(qid)
    m = df[df["query_id"].astype(str) == qid_str]
    if m.empty:
        raise KeyError(f"query_id not found in gold set: {qid_str}")
    return dict(m.iloc[0].to_dict())


In [ ]:
# --- Build pipeline (dev) ---
from compliancegpt.pipeline.pipeline import ComplianceGPTPipeline

def build_pipeline(
    *,
    framework_version: str,
    use_qur_live: bool,
    use_org_profile: bool,
    verify_strict_extras: bool = False,
    verify_strict_verbatim: bool = True,
    verify_strict_version: bool = False,
) -> ComplianceGPTPipeline:
    fw = framework_version.lower().strip()
    if fw not in {"rev4", "rev5"}:
        raise ValueError(f"framework_version must be rev4 or rev5 (got {framework_version!r})")

    ccs_path = REV5_CATALOG_PATH if fw == "rev5" else REV4_CATALOG_PATH
    org_profile = (ORG_PROFILE_REV5 if fw == "rev5" else ORG_PROFILE_REV4) if use_org_profile else None
    if org_profile and (not os.path.exists(org_profile)):
        print(f"[WARN] org_profile not found; continuing without profile: {org_profile}")
        org_profile = None

    pipe = ComplianceGPTPipeline(
        framework_version=fw,
        ccs_path=ccs_path,
        use_qur=bool(use_qur_live),
        org_profile_path=org_profile,
        verify_strict_extras=bool(verify_strict_extras),
        verify_strict_verbatim=bool(verify_strict_verbatim),
        verify_strict_version=bool(verify_strict_version),
    )
    return pipe


In [ ]:
# --- Output utilities (dev) ---
import json

def print_contract_summary(out: Dict[str, Any]) -> None:
    c = out.get("contract", out)
    print("status:", c.get("status"))
    print("evidence_spans_len:", len(c.get("evidence_spans") or []))
    print("odp_required_list:", c.get("odp_required_list") or [])
    if "verifier_ran" in c:
        print("verifier_ran:", c.get("verifier_ran"))
    print("verifier_pass:", c.get("verifier_pass"))
    errs = c.get("verifier_errors") or []
    if errs:
        print("verifier_errors:", errs[:10])

def dump_contract_json(out: Dict[str, Any], *, max_chars: int = 5000) -> None:
    c = out.get("contract", out)
    s = json.dumps(c, ensure_ascii=False, indent=2)
    print(s[:max_chars] + ("\n... (truncated) ..." if len(s) > max_chars else ""))


## Configure and run

In [ ]:
# === DEV CONFIG ===
FRAMEWORK_VERSION = "rev5"      # "rev4" or "rev5"
QUERY_MODE = "freeform"         # "freeform" or "gold_id"

# Freeform query
QUERY_TEXT = "When must account managers and the designated personnel be notified about user account changes?"

# Gold query (by id)
GOLD_QUERY_ID = 1               # 1..100 for rev5; 1..36 for rev4

# Rewrites
USE_QUR_PRECOMPUTED = True
USE_QUR_LIVE = False

# Org profile
USE_ORG_PROFILE = False

# Verification strictness knobs
VERIFY_STRICT_EXTRAS = False
VERIFY_STRICT_VERBATIM = True
VERIFY_STRICT_VERSION = False

TOP_K = 12
SHOW_FULL_CONTRACT = True


In [ ]:
pipe = build_pipeline(
    framework_version=FRAMEWORK_VERSION,
    use_qur_live=USE_QUR_LIVE,
    use_org_profile=USE_ORG_PROFILE,
    verify_strict_extras=VERIFY_STRICT_EXTRAS,
    verify_strict_verbatim=VERIFY_STRICT_VERBATIM,
    verify_strict_version=VERIFY_STRICT_VERSION,
)
print("[OK] Pipeline initialized.")

In [ ]:
gold_df = load_gold(FRAMEWORK_VERSION)

if QUERY_MODE == "gold_id":
    gold_row = get_gold_row(gold_df, GOLD_QUERY_ID)
    query = str(gold_row["original_query"])
    qid = str(gold_row["query_id"])
else:
    gold_row = None
    query = str(QUERY_TEXT).strip()
    qid = None

rewrites = []
if USE_QUR_PRECOMPUTED and qid is not None:
    rewrites = load_qur_map(FRAMEWORK_VERSION).get(qid, [])

RUN_VERIFY = bool(gold_row is not None)  # IMPORTANT: verify only in evaluation mode

print("query_id:", qid)
print("query:", query)
print("n_rewrites:", len(rewrites))
print("run_verify:", RUN_VERIFY)


In [ ]:
out = pipe.answer(
    query,
    top_k=int(TOP_K),
    rewrites=rewrites if rewrites else None,
    gold_row=gold_row,
    use_generator=True,
    run_verify=RUN_VERIFY,
)

print_contract_summary(out)

if SHOW_FULL_CONTRACT:
    dump_contract_json(out, max_chars=5000)
